# 局部 VAE 条件写入头＋软状态空间同步＋32位纠错载荷

B 线直接 Run all 入口，自动加载可用源码，无需上传源码 ZIP。此 Notebook 会执行用户确认的唯一固定训练与 CAL/DEV 验证；当前交付只经过 CPU/static 与独审，真实效果待本次运行。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 一次尝试与固定预算

已有云端或本地执行记录即只读，失败或中断不重启训练。模型、原视频缓存、VAE 特征和 terminal 仅存 /content；Drive 保留小头、必要媒体、原始 logits、固定任务和失败记录。不得同时开两个运行时启动这个目录。


In [ ]:

from pathlib import Path
import json,os,shutil,stat,subprocess,sys,tempfile,urllib.request,zipfile,traceback
SCRATCH=Path('/content/video-wm-local-vae-head-v1')
LOCAL=SCRATCH/'results'
OUTPUT=Path('/content/drive/MyDrive/Video-WM/B-Local-VAE-Head-V1/fixed-six-source-v1')
SOURCE_REF='dev/local-joint-state-payload-v1'
SOURCE_DIRECTORY=None
SOURCE_ROOT=None
PREPARATION_ERROR=None
CLIENT=None
CAN_START=False
os.environ.update(HF_HOME='/content/video-wm-cache/huggingface',HF_HUB_CACHE='/content/video-wm-cache/huggingface/hub',TORCH_HOME='/content/video-wm-cache/torch')
"""Ordinary Drive storage and readable once-attempt receipt; no model work."""
from pathlib import Path
import json
import tempfile
import time

COMPONENTS=('Video-WM','B-Local-VAE-Head-V1','fixed-six-source-v1')
FOLDER='application/vnd.google-apps.folder'
API='https://www.googleapis.com/drive/v3/files'

def quoted(value):return "'"+str(value).replace('\\','\\\\').replace("'","\\'")+"'"

class Drive:
    def __init__(self,session):self.session=session
    def request(self,method,url,**kwargs):
        response=self.session.request(method,url,timeout=(20,180),**kwargs)
        response.raise_for_status();return response
    def children(self,parent):
        result=[];token=None
        while True:
            params=dict(q='trashed = false and '+quoted(parent)+' in parents',fields='nextPageToken,incompleteSearch,files(id,name,mimeType,size)',pageSize=1000)
            if token:params['pageToken']=token
            with self.request('GET',API,params=params) as r:value=r.json()
            if value.get('incompleteSearch'):raise IOError('incomplete Drive directory listing')
            result+=value.get('files',[]);token=value.get('nextPageToken')
            if not token:return result
    def metadata(self,ident):
        with self.request('GET',API+'/'+ident,params={'fields':'id,name,mimeType,size'}) as r:return r.json()
    def folder(self,components=COMPONENTS,create=False):
        parent='root'
        for name in components:
            found=[m for m in self.children(parent) if m['name']==name]
            if len(found)>1:raise ValueError('ambiguous Drive folder '+name)
            if found:
                if found[0]['mimeType']!=FOLDER:raise ValueError('expected Drive folder '+name)
                parent=found[0]['id']
            elif create:
                with self.request('POST',API,json=dict(name=name,mimeType=FOLDER,parents=[parent]),params={'fields':'id'}) as r:parent=r.json()['id']
            else:return None
        return parent
    def read_to(self,meta,destination):
        path=Path(destination);path.parent.mkdir(parents=True,exist_ok=True);n=0
        with self.request('GET',API+'/'+meta['id'],params={'alt':'media'},headers={'Accept-Encoding':'identity'},stream=True) as r:
            with path.open('wb') as stream:
                for chunk in r.iter_content(1024*1024):
                    if chunk:stream.write(chunk);n+=len(chunk)
        if n==0 or ('size' in meta and n!=int(meta['size'])):raise IOError('empty/incomplete actual Drive blob read')
        return n
    def upload(self,parent,path,existing_id=None,name=None):
        """One resumable upload, no automatic retry after an uncertain write."""
        path=Path(path);metadata={'name':name or path.name}
        if existing_id is None:metadata['parents']=[parent]
        url='https://www.googleapis.com/upload/drive/v3/files'+('/'+existing_id if existing_id else '')
        with self.request('PATCH' if existing_id else 'POST',url,params={'uploadType':'resumable','fields':'id,name,size'},
            json=metadata,headers={'X-Upload-Content-Length':str(path.stat().st_size),'X-Upload-Content-Type':'application/octet-stream'}) as r:
            location=r.headers['Location']
        with path.open('rb') as stream:
            with self.request('PUT',location,data=stream,headers={'Content-Length':str(path.stat().st_size),'Content-Type':'application/octet-stream'}) as r:return r.json()

def authenticate():
    from google.colab import auth
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    auth.authenticate_user()
    credentials,_=google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
    return Drive(AuthorizedSession(credentials))

def preflight(client,mounted):
    folder=client.folder();cloud=client.children(folder) if folder else []
    local=list(Path(mounted).iterdir()) if Path(mounted).is_dir() else []
    # Any actual result artifact consumes this candidate's once-attempt scope.
    existing=bool(cloud or local)
    return dict(status='EXISTING_ATTEMPT' if existing else 'NO_EXECUTION_OBSERVED',can_start=not existing,
        folder_id=folder,cloud_names=[m['name'] for m in cloud],mounted_names=[p.name for p in local])

def reserve(client,mounted):
    state=preflight(client,mounted)
    if not state['can_start']:return dict(status='NOT_STARTED',state=state)
    folder=client.folder(create=True)
    payload=dict(status='STARTED_ONE_FIXED_ATTEMPT',unix=time.time(),retry=False,atomic_mutex=False,
        purpose='local VAE head fixed training/CAL/DEV; interrupted attempt is not restarted')
    try:
        with tempfile.TemporaryDirectory(prefix='vae-start-') as tmp:
            path=Path(tmp)/'repeat_start.json';path.write_text(json.dumps(payload))
            item=client.upload(folder,path);client.read_to(item,path)
            if json.loads(path.read_text()).get('status')!='STARTED_ONE_FIXED_ATTEMPT':raise IOError('unreadable startup receipt')
        mounted=Path(mounted);mounted.mkdir(parents=True,exist_ok=True)
        (mounted/'repeat_start.json').write_text(json.dumps(payload))
        return dict(status='START_RECEIPT_CLOUD_READ_BACK',folder_id=folder,file_id=item['id'],atomic_mutex=False)
    except BaseException as e:
        mounted=Path(mounted);mounted.mkdir(parents=True,exist_ok=True)
        (mounted/'repeat_start_unknown.json').write_text(json.dumps(dict(error=f'{type(e).__name__}: {e}',retry=False)))
        raise

def download_training(client,cfg,scratch):
    result={}
    for row in cfg['training_sources']:
        target=Path(scratch)/'inputs'/(row['id']+'.mp4')
        meta=client.metadata(row['drive_file_id']);client.read_to(meta,target)
        result[row['id']]=str(target)
    return result

def sync_results(client,local,mounted,progress=print):
    """Publish saved artifacts then read actual blobs. Does not change method status."""
    local=Path(local);root=client.folder(create=True);folders={'':root};indices={};report=dict(uploaded=0,readable=0,media=0,errors=[],missing=[],method_status_modified=False,physical_calls=0)
    def parent_for(parts):
        parent=root;prefix=''
        for name in parts:
            prefix='/'.join(filter(None,(prefix,name)))
            if prefix not in folders:
                found=[m for m in client.children(parent) if m['name']==name]
                if len(found)>1:raise ValueError('ambiguous result subfolder '+prefix)
                if found:
                    if found[0]['mimeType']!=FOLDER:raise ValueError('result path is not folder '+prefix)
                    folders[prefix]=found[0]['id']
                else:
                    with client.request('POST',API,json=dict(name=name,mimeType=FOLDER,parents=[parent]),params={'fields':'id'}) as r:folders[prefix]=r.json()['id']
            parent=folders[prefix]
        return parent
    files=[p for p in local.rglob('*') if p.is_file() and not p.name.endswith('.tmp')]
    # Final core records are uploaded last, after observations/media/checkpoint.
    files.sort(key=lambda p:(p.name in ('result.json','evaluation.json'),str(p)))
    with tempfile.TemporaryDirectory(prefix='vae-readback-') as tmp:
        for number,path in enumerate(files,1):
            rel=path.relative_to(local)
            try:
                parent=parent_for(rel.parts[:-1])
                if parent not in indices:indices[parent]=client.children(parent)
                matches=[m for m in indices[parent] if m['name']==path.name]
                if len(matches)>1:raise ValueError('ambiguous existing result blob '+str(rel))
                item=client.upload(parent,path,matches[0]['id'] if matches else None)
                report['uploaded']+=1;target=Path(tmp)/'readback';client.read_to(item,target);report['readable']+=1
                if path.suffix=='.mp4':report['media']+=1
                if path.name in ('result.json','evaluation.json'):
                    report[path.name]=json.loads(target.read_text()) if path.name=='result.json' else {k:json.loads(target.read_text()).get(k) for k in ('status','method_status','recommendation')}
            except Exception as e:report['errors'].append(dict(file=str(rel),error=f'{type(e).__name__}: {e}'))
            if number%20==0 or number==len(files):progress(f'Cloud saved/read: {number}/{len(files)}')
    for required in ('result.json','evaluation.json','plan.json','attempt.json','physical_counts.json','quality.json'):
        if not (local/required).is_file():report['missing'].append(required)
    report['status']='SAVED_ARTIFACTS_CLOUD_READABLE' if not report['errors'] and not report['missing'] else 'CLOUD_SAVE_UNCONFIRMED'
    report['expected_full_budget_media']=72;report['fixed_media_missing']=72-report['media']
    (Path(mounted)/'cloud_save_report.json').write_text(json.dumps(report,ensure_ascii=False,allow_nan=False))
    return report

def read_existing(client,mounted):
    folder=client.folder();result=dict(status='READ_ONLY_EXISTING_ATTEMPT',physical_calls=0,errors=[])
    if folder is None:
        result['status']='NO_CLOUD_ATTEMPT_RECORD'
        result['errors'].append(dict(file='output_folder',error='cloud output folder missing; preparation may not have started a scientific attempt'))
        result['fixed_recovery']=recovery_snapshot({})
        result['result.json']=dict(status='NO_SAVED_COMPUTATION',stop='cloud output folder missing; no completed computation claimed',
            denominators=result['fixed_recovery']['denominators'],physical_counts=None)
        result['evaluation.json']=dict(status='UNRESOLVED',method_status='NOT_EVALUATED',recommendation='RESOLVE_RECORDED_PREPARATION_ERROR',per_source={},failures=[])
        return result
    children=client.children(folder)
    with tempfile.TemporaryDirectory(prefix='vae-existing-') as tmp:
        for name in ('result.json','evaluation.json','physical_counts.json','task_state.json','plan.json','quality.json'):
            found=[m for m in children if m['name']==name]
            try:
                if len(found)!=1:raise FileNotFoundError('expected one readable '+name)
                path=Path(tmp)/name;client.read_to(found[0],path);value=json.loads(path.read_text())
                result[name]=value if name!='evaluation.json' else {k:value.get(k) for k in ('status','method_status','recommendation','CAL','DEV','summaries','per_source','failures','denominators')}
            except Exception as e:result['errors'].append(dict(file=name,error=f'{type(e).__name__}: {e}'))
    result['fixed_recovery']=recovery_snapshot(result)
    if 'result.json' not in result:
        result['result.json']=dict(status='UNRESOLVED_INTERRUPTED_ATTEMPT',stop='final record absent; no scientific operation repeated',
            denominators=result['fixed_recovery']['denominators'],physical_counts=result.get('physical_counts.json'))
    if 'evaluation.json' not in result:
        result['evaluation.json']=dict(status='UNRESOLVED',method_status='UNRESOLVED_INTERRUPTED_ATTEMPT',recommendation='NO_RETRY_THIS_ATTEMPT',
            per_source=result['fixed_recovery']['per_source'],failures=result['fixed_recovery']['tasks'],
            null_zero_criteria=None,literal_error_count=None)
    return result

def recovery_snapshot(values):
    """Summarize already saved execution states, without scoring, BCH, or truth reads."""
    state=values.get('task_state.json',{});tasks=state.get('tasks',[]);plan=values.get('plan.json',{})
    if not tasks and plan:
        for reader,key in (('STATE_SPACE','main'),('I','I')):
            tasks.extend(dict(row,reader=reader,plan_status='UNRESOLVED',read_status='UNRESOLVED',decision_status='UNRESOLVED') for row in plan.get(key,[]))
        originals={r['task']:r for r in plan.get('main',[])}
        for row in plan.get('post_OWU',[]):
            tasks.append(dict(originals.get(row['task'],row),reader=row['mode'],plan_status='POST_ONLY',read_status='UNRESOLVED',decision_status='UNRESOLVED'))
    per_source={}
    for row in tasks:
        key=row['reader']+'/'+row.get('source','UNKNOWN');summary=per_source.setdefault(key,dict(planned_tasks=0,returned_reads=0,not_started=0,unresolved=0,information_slots=0,known_outputs=0,known_erasures=0,literal_errors=None,null_zero_criteria=None))
        summary['planned_tasks']+=1;summary['information_slots']+=row.get('information_slots',32)
        status=row.get('read_status');summary['returned_reads']+=status=='RETURNED';summary['not_started']+=status=='NOT_STARTED'
        summary['unresolved']+=status not in ('NOT_STARTED','RETURNED')
        summary['known_outputs']+=row.get('decision_status')=='OUTPUT';summary['known_erasures']+=row.get('decision_status')=='ERASURE'
    denominators=state.get('denominators') or dict(main_tasks=150,main_information_slots=4800,I_tasks=150,I_information_slots=4800,post_OWU_tasks=90,post_OWU_information_slots=2880,quality_pairs=42,quality_frames=6702,quality_adjacent=6660)
    return dict(status='SAVED_EXECUTION_STATES_ONLY',denominators=denominators,tasks=tasks,per_source=per_source,
        quality=values.get('quality.json'),physical_counts=values.get('physical_counts.json'),
        missing_state_rows=max(390-len(tasks),0),unread_is_not_zero=True,scientific_calls_this_recovery=0,
        actual_errors_and_temporal_attribution='UNKNOWN without saved final evaluation')

try:
    CLIENT=authenticate()
    STATE=preflight(CLIENT,OUTPUT)
    local_existing=LOCAL.exists() and any(LOCAL.iterdir())
    CAN_START=STATE['can_start'] and not local_existing
    print('一次尝试检查:',STATE['status'],'本地执行记录:',bool(local_existing))
except Exception as error:
    PREPARATION_ERROR=f'{type(error).__name__}: {error}'
    print('Drive读取实际失败，未启动:',PREPARATION_ERROR)


## 直接加载 B 线源码

固定 TRAIN n01–n04 既存 OFF/full；CAL m01/m02，DEV m03/m04/v02/v04。保留旧 prompt/seed。代码和依赖的来源记录不构成摘要或版本门禁。


In [ ]:

if CAN_START:
    try:
        SCRATCH.mkdir(parents=True,exist_ok=True)
        workspace=Path(tempfile.mkdtemp(prefix='source-',dir=SCRATCH)).resolve()
        if SOURCE_DIRECTORY is None:
            url='https://codeload.github.com/RICHAAARC/SC-SSTW/zip/refs/heads/'+SOURCE_REF
            archive=workspace/'source.zip';urllib.request.urlretrieve(url,archive)
            destination=workspace/'source';destination.mkdir()
            with zipfile.ZipFile(archive) as zipped:
                for member in zipped.infolist():
                    normalized=member.filename.replace('\\','/')
                    target=(destination/normalized).resolve()
                    if normalized.startswith('/') or any(':' in p for p in normalized.split('/')) or not target.is_relative_to(destination.resolve()) or stat.S_ISLNK(member.external_attr>>16):raise ValueError('unsafe source archive path')
                zipped.extractall(destination)
            entries=list(destination.glob('*/experiments/wan_state_clock/b_local_vae_head_v1_run.py'))
            if len(entries)!=1:raise FileNotFoundError('available B source lacks local VAE entry: '+url)
            SOURCE_ROOT=entries[0].parents[2]
        else:
            SOURCE_ROOT=workspace/'source'
            shutil.copytree(Path(SOURCE_DIRECTORY).expanduser(),SOURCE_ROOT,ignore=shutil.ignore_patterns('.git','__pycache__'))
        CONFIG=SOURCE_ROOT/'experiments/wan_state_clock/configs/b_local_vae_head_v1.json'
        CFG=json.loads(CONFIG.read_text())
        print('可用源码:',SOURCE_ROOT,'固定预算:',CFG['physical_budget'])
    except Exception as error:
        CAN_START=False;PREPARATION_ERROR=f'{type(error).__name__}: {error}'
        print('源码准备失败:',PREPARATION_ERROR)
else:print('已有记录或云端读取未决：跳过源码准备与真实执行。')


## 复用环境，执行一次

沿用 Wan/VideoSeal 导入修复、FP32 VAE 和 H264 CRF18 媒体链。3000 Adam 步（首20步资源观察已包含），batch8、lr1e-4。E181=4、D181=16；编码3072、解码3076、probe76，训练与验证分账。CAL10正例全部正确且40null零检出/输出才启动DEV。没有参数扫描或自动追加新源实验。


In [ ]:
if CAN_START:
    try:

        sys.path.insert(0,str(SOURCE_ROOT))
        from runtime.wan.b_local_vae_head_setup_v1 import prepare_vendor,environment as source_environment,check_dependencies
        VENDOR=prepare_vendor(SCRATCH)
        environment=source_environment(SOURCE_ROOT,VENDOR)
        LOG = SCRATCH/'setup.log'
        def command(args, label, check=True):
            print(label, flush=True)
            with LOG.open('a') as stream:
                stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
                completed = subprocess.run(args,cwd=VENDOR,env=environment,stdout=stream,stderr=subprocess.STDOUT)
            if completed.returncode and check:
                raise RuntimeError(label+' failed; see '+str(LOG))
            return completed.returncode
        def ensure_imports(probe, packages, label):
            import re
            mapping = {p.replace('-', '_'):p for p in packages}; mapping.update({'cv2':'opencv-python','pywt':'PyWavelets','skimage':'scikit-image','yaml':'PyYAML'})
            repaired = set()
            while True:
                offset = LOG.stat().st_size if LOG.exists() else 0
                if not command([sys.executable, '-c', probe], 'Probe '+label, check=False):
                    return
                with LOG.open('rb') as stream:
                    stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
                missing = re.search("ModuleNotFoundError: No module named [\'\"]([^\'\"]+)[\'\"]", failure)
                if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                    action = 'remove incompatible optional torchao'
                    args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
                elif missing and missing.group(1).split('.')[0] in mapping:
                    package = mapping[missing.group(1).split('.')[0]]
                    action = 'install missing '+package
                    args = [sys.executable, '-m', 'pip', 'install', package]
                else:
                    raise RuntimeError(label+' actual import failure; see '+str(LOG))
                if action in repaired:
                    raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
                repaired.add(action)
                command(args, action)
        ensure_imports('import numpy,torch', ['numpy','torch'], 'tensor dependencies')
        ensure_imports('import accelerate,safetensors,huggingface_hub,transformers,sentencepiece,ftfy; from diffusers import WanPipeline,AutoencoderKLWan',
            ['diffusers','accelerate','safetensors','huggingface-hub','transformers','sentencepiece','ftfy'], 'Wan pipeline and VAE')
        import shutil
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            command(['apt-get','-qq','update'], 'Prepare media dependency')
            command(['apt-get','-qq','install','-y','ffmpeg'], 'Install FFmpeg')
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            raise RuntimeError('ffmpeg/ffprobe unavailable; see '+str(LOG))

        ensure_imports('import numpy,torch,torchvision; import videoseal.utils.cfg', ['numpy','torch','torchvision','pandas','omegaconf','einops','lpips','timm','pycocotools','tqdm','pytorch-msssim','tensorboard','calflops','transformers','safetensors','scipy','ffmpeg-python','av','huggingface-hub','ftfy','sentencepiece','accelerate'], 'VideoSeal cfg import; no model loading')

        probe='import main.tube_state.b_local_vae_head_v1; import runtime.wan.b_local_vae_head_backend_v1; import experiments.wan_state_clock.b_local_vae_head_v1_run'
        command([sys.executable,'-c',probe],'Probe actual writer/backend/runner imports')
        check_dependencies(SOURCE_ROOT,VENDOR)
        INPUTS=download_training(CLIENT,CFG,SCRATCH)
        INPUT_FILE=SCRATCH/'downloaded_inputs.json';INPUT_FILE.write_text(json.dumps(INPUTS))
        START=reserve(CLIENT,OUTPUT)
        print('启动收据:',START)
        if START['status']=='START_RECEIPT_CLOUD_READ_BACK':
            args=[sys.executable,'-u','-c','from experiments.wan_state_clock.b_local_vae_head_v1_run import _notebook_reserved_main; raise SystemExit(_notebook_reserved_main())','--config',str(CONFIG),'--local-output',str(LOCAL),'--mirror',str(OUTPUT),'--inputs',str(INPUT_FILE)]
            with (SCRATCH/'runner.log').open('a') as stream:
                worker=subprocess.Popen(args,cwd=SOURCE_ROOT,env=environment,stdout=stream,stderr=subprocess.STDOUT)
                try:
                    while worker.poll() is None:
                        try:worker.wait(timeout=30)
                        except subprocess.TimeoutExpired:
                            try:print(json.loads((LOCAL/'progress.json').read_text()),flush=True)
                            except (OSError,ValueError):print('同一 worker 运行中；详见',SCRATCH/'runner.log',flush=True)
                except BaseException:
                    worker.terminate()
                    try:worker.wait(timeout=10)
                    except subprocess.TimeoutExpired:worker.kill();worker.wait()
                    raise
            print('Runner exit:',worker.returncode)
            if worker.returncode:
                print('\n'.join((SCRATCH/'runner.log').read_text(errors='replace').splitlines()[-40:]))
        else:print('没有取得可读启动收据，本次未启动。')

    except BaseException as error:
        PREPARATION_ERROR=f'{type(error).__name__}: {error}'
        print('准备或 worker 未完成:',PREPARATION_ERROR)
        if (SCRATCH/'setup.log').exists():print('\n'.join((SCRATCH/'setup.log').read_text(errors='replace').splitlines()[-40:]))
    CAN_START=False
else:print('只读已有尝试；不安装、不加载模型、不补跑。')


## 固定结果与保存核验

主读法150任务/4800槽，I独立基线另150/4800；后评O/W_GROUP/U另90/2880。全部计划保存后读payload，全部blind记录后才后评。42质量对/6702帧/6660相邻对，无感知PASS。原始逐帧与8帧窗口logits、直接观察/ECC修复、错误时间引用、未决项全部保留。云端可读与方法通过分别报告。


In [ ]:

if PREPARATION_ERROR:print('实际错误:',PREPARATION_ERROR)
if CLIENT is not None:
    if LOCAL.is_dir() and any(LOCAL.iterdir()):
        SAVE=sync_results(CLIENT,LOCAL,OUTPUT)
        print('云端保存:',{k:SAVE[k] for k in ('status','uploaded','readable','media','fixed_media_missing','errors','missing','physical_calls')})
    SAVED=read_existing(CLIENT,OUTPUT)
    print('云端读取错误:',SAVED['errors'])
    result=SAVED.get('result.json',{});evaluation=SAVED.get('evaluation.json',{})
    print('计算状态:',result.get('status'),'停止:',result.get('stop'))
    print('固定分母:',result.get('denominators'))
    print('实际调用:',result.get('physical_counts'))
    print('方法:',evaluation.get('method_status'),'推荐:',evaluation.get('recommendation'))
    print('中断恢复固定逐项状态:',{k:SAVED['fixed_recovery'][k] for k in ('status','missing_state_rows','scientific_calls_this_recovery')})
    print('逐源:',json.dumps(evaluation.get('per_source'),ensure_ascii=False,indent=2))
    print('失败/未决数量:',len(evaluation.get('failures',[])))
print('完整结果目录:',OUTPUT)
print('一次训练/固定已见来源证据；ECC不是来源认证或逐位物理支持。未宣称总体FPR、感知质量PASS或新源独立确认。')
